# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

> Dataset title and description are loaded from metadata in Section 1 below.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Title: {metadata.name}\n")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entity references use their `@id` as required.

Below, we enumerate all record sets, their `@id`, their available fields, and the associated field IDs.


In [ ]:
# Show overview of all record sets and their fields, referencing by @id
from pprint import pprint

record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in this dataset.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs.name}")
        print(f"  @id: {rs['@id']}")
        print(f"  Description: {rs.description if hasattr(rs, 'description') else 'No description'}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field['@id']}, type: {field.data_type})")
        print()

### Record example preview
Below, see an example of a few records for a selected record set (referenced by `@id`).


In [ ]:
# If record sets exist, preview some records by @id
rs_ids = [rs['@id'] for rs in dataset.record_sets]
if rs_ids:
    example_record_set_id = rs_ids[0]  # First record set
    print(f"Previewing first records in record set '{example_record_set_id}':\n")
    for i, rec in enumerate(dataset.records(record_set=example_record_set_id)):
        pprint(rec)
        if i >= 2:
            break
else:
    print("No record sets available to preview records.")

## 3. Data Extraction
Load data from specific record set(s) into DataFrames for analysis. Record set and field references all use their respective `@id`s as required by the dataset schema.


In [ ]:
# Extract all available record sets into pandas DataFrames by their @id
record_sets = list(dataset.record_sets)
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # Retrieve all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

if dataframes:
    # Show the first available DataFrame columns and head
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"Columns for record set '{selected_record_set_id}':")
    print(dataframes[selected_record_set_id].columns.tolist())

    print(dataframes[selected_record_set_id].head())
else:
    print("No tabular data extracted. Did you run this cell after Section 1?")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria (for example, proteins with molecular weight over a threshold), normalizing numeric fields, and grouping or summarizing data. We reference fields by their `@id` only.


In [ ]:
# Select a record set and numeric field @id for demonstration
if dataframes:
    # Pick the first record set with numeric columns
    df = None
    selected_record_set_id = None
    numeric_field_id = None

    for rid, _df in dataframes.items():
        # Find numeric fields by dtype or by name (commonly MW, molecular weight, or similar)
        for col in _df.columns:
            if pd.api.types.is_numeric_dtype(_df[col]) or 'weight' in col.lower() or 'mw' in col.lower():
                df = _df
                selected_record_set_id = rid
                numeric_field_id = col
                break
        if df is not None:
            break

    if df is not None and numeric_field_id is not None:
        print(f"Operating on record set: {selected_record_set_id}\nNumeric field for analysis: {numeric_field_id}\n")
        
        # Drop missing values in the numeric field
        metric = df[numeric_field_id].dropna()

        # Filter: Keep records with value > threshold (e.g. MW>10000)
        threshold = metric.mean() if metric.mean()>0 else 10000
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with '{numeric_field_id}' > {threshold:.2f} (filter based on mean or 10000 if mean<=0):")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - metric.mean()
        ) / (metric.std() if metric.std()>0 else 1)
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another field (e.g. by description or annotated type)
        group_field_candidates = [c for c in df.columns if c != numeric_field_id and df[c].dtype == 'O']
        if group_field_candidates:
            group_field = group_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by '{group_field}':")
            print(grouped_df.head())
        else:
            print("\nNo suitable categorical field for grouping found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No tabular data available for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field and, where possible, comparisons by groupings (e.g. protein types or annotations).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if 'filtered_df' in locals() and not filtered_df.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group_field exists, plot grouped boxplot
    if 'group_field' in locals():
        plt.figure(figsize=(10, 4))
        # Limit to first 10 groups for clarity
        vals = (
            filtered_df.groupby(group_field)[numeric_field_id]
            .mean()
            .sort_values(ascending=False)
            .head(10)
        )
        sns.barplot(x=vals.index, y=vals.values)
        plt.title(f"Mean {numeric_field_id} by '{group_field}' (Top 10)")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No data to visualize.")

## 6. Conclusion

- We demonstrated exploration and processing of a FAIR-compliant mass spectrometry dataset using the `mlcroissant` library, referencing all record sets, fields, and columns by their `@id`s.
- Data was loaded, overviewed, basic filtering and normalization were performed, and key statistics and visualizations were displayed for a selected numeric field.
- This workflow can be generalized for any Croissant dataset with record sets and tabular data.
